In [1]:
pip install -q torch torchvision transformers datasets scikit-learn matplotlib seaborn pandas


In [3]:
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
import torchvision
import torchvision.transforms as transforms
from torchvision.models import resnet18
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns

# ---------------------------------------------------------
# 1. Reproducibility & Device Setup
# ---------------------------------------------------------
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs("results", exist_ok=True)
print(f"Running on device: {device}")

# ---------------------------------------------------------
# 2. Controlled Label Noise Injection
# ---------------------------------------------------------
def inject_label_noise(labels, num_classes, noise_rate=0.2, noise_type="symmetric"):
    labels = np.array(labels)
    n_samples = len(labels)
    noisy_labels = labels.copy()
    n_noisy = int(noise_rate * n_samples)
    noisy_indices = np.random.choice(n_samples, size=n_noisy, replace=False)
    is_corrupted = np.zeros(n_samples, dtype=int)

    for idx in noisy_indices:
        orig = labels[idx]
        if noise_type == "symmetric":
            choices = [c for c in range(num_classes) if c != orig]
            noisy_labels[idx] = np.random.choice(choices)
        elif noise_type == "asymmetric":
            noisy_labels[idx] = (orig + 1) % num_classes
        is_corrupted[idx] = 1

    return noisy_labels, is_corrupted

# ---------------------------------------------------------
# 3. Indexed Dataset Wrapper for Sample-Level Tracking
# ---------------------------------------------------------
class IndexedDataset(Dataset):
    def __init__(self, base_dataset, noisy_labels):
        self.base_dataset = base_dataset
        self.noisy_labels = noisy_labels

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, idx):
        item = self.base_dataset[idx]
        if isinstance(item, tuple):
            x, _ = item
            return x, int(self.noisy_labels[idx]), idx
        else:
            return item["input_ids"], item["attention_mask"], int(self.noisy_labels[idx]), idx

# ---------------------------------------------------------
# 4. Training Dynamics Tracker (Confidence & Variability)
# ---------------------------------------------------------
def compute_cartography_metrics(prob_history):
    """
    prob_history: np.ndarray of shape (num_epochs, num_samples)
    """
    confidence = np.mean(prob_history, axis=0)
    variability = np.std(prob_history, axis=0)
    # Anomaly score: low confidence + high loss proxy
    suspicion_score = 1.0 - confidence
    return confidence, variability, suspicion_score

def plot_cartography(confidence, variability, is_corrupted, title, save_path):
    plt.figure(figsize=(7, 5), dpi=300)
    df = pd.DataFrame({
        "Confidence": confidence,
        "Variability": variability,
        "Status": np.where(is_corrupted == 1, "Corrupted Label", "Clean Label")
    })
    sns.scatterplot(
        data=df.sample(min(3000, len(df)), random_state=42),
        x="Variability", y="Confidence", hue="Status",
        palette={"Clean Label": "#2b5c8f", "Corrupted Label": "#d95f02"},
        alpha=0.65, s=18
    )
    plt.title(title, fontsize=12, fontweight="bold")
    plt.xlabel("Variability (Std. Dev. of Assigned Class Prob.)")
    plt.ylabel("Confidence (Mean of Assigned Class Prob.)")
    plt.legend(frameon=True)
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()

# ---------------------------------------------------------
# 5. Modality A: CIFAR-10 with Modified ResNet-18
# ---------------------------------------------------------
def get_cifar10_resnet18(num_classes=10):
    model = resnet18(weights=None, num_classes=num_classes)
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    return model.to(device)

def evaluate_vision(model, loader):
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            out = model(x)
            preds.extend(out.argmax(dim=1).cpu().numpy())
            targets.extend(y.numpy())
    return accuracy_score(targets, preds)

def run_cifar_experiment(noise_rate=0.2, noise_type="symmetric", epochs=15):
    print(f"\n--- Running CIFAR-10 Experiment ({noise_rate*100:.0f}% {noise_type} noise) ---")
    transform_train = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ])
    transform_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ])

    raw_train = torchvision.datasets.CIFAR10(root="./data", train=True, download=True, transform=transform_train)
    test_set = torchvision.datasets.CIFAR10(root="./data", train=False, download=True, transform=transform_test)

    # Use 15,000 samples for fast 1-day execution (change to 50000 for full run)
    sub_indices = list(range(15000))
    train_subset = Subset(raw_train, sub_indices)
    clean_targets = [raw_train.targets[i] for i in sub_indices]

    noisy_labels, is_corrupted = inject_label_noise(clean_targets, 10, noise_rate, noise_type)
    indexed_train = IndexedDataset(train_subset, noisy_labels)

    train_loader = DataLoader(indexed_train, batch_size=128, shuffle=True, num_workers=2)
    test_loader = DataLoader(test_set, batch_size=256, shuffle=False, num_workers=2)

    model = get_cifar10_resnet18(10)
    optimizer = torch.optim.SGD(model.parameters(), lr=0.05, momentum=0.9, weight_decay=5e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    prob_history = np.zeros((epochs, len(indexed_train)), dtype=np.float32)

    for epoch in range(epochs):
        model.train()
        for x, y, idx in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(x)
            loss = F.cross_entropy(logits, y)
            loss.backward()
            optimizer.step()

            probs = F.softmax(logits.detach(), dim=1)
            assigned_probs = probs[torch.arange(len(y)), y].cpu().numpy()
            prob_history[epoch, idx.numpy()] = assigned_probs
        scheduler.step()

    noisy_acc = evaluate_vision(model, test_loader)
    conf, var, suspicion = compute_cartography_metrics(prob_history)

    auroc = roc_auc_score(is_corrupted, suspicion)
    prauc = average_precision_score(is_corrupted, suspicion)
    plot_cartography(conf, var, is_corrupted, f"CIFAR-10 Training Dynamics ({noise_type})", "results/cifar_cartography.png")

    # Prune top suspicious samples and retrain
    keep_mask = np.argsort(suspicion)[:int((1.0 - noise_rate) * len(indexed_train))]
    pruned_subset = Subset(indexed_train, keep_mask)
    pruned_loader = DataLoader(pruned_subset, batch_size=128, shuffle=True, num_workers=2)

    clean_model = get_cifar10_resnet18(10)
    optimizer = torch.optim.SGD(clean_model.parameters(), lr=0.05, momentum=0.9, weight_decay=5e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    for epoch in range(epochs):
        clean_model.train()
        for x, y, _ in pruned_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            loss = F.cross_entropy(clean_model(x), y)
            loss.backward()
            optimizer.step()
        scheduler.step()

    pruned_acc = evaluate_vision(clean_model, test_loader)
    return {"Modality": "Vision (CIFAR-10)", "AUROC": auroc, "PR-AUC": prauc, "Noisy_Acc": noisy_acc, "Pruned_Acc": pruned_acc}

# ---------------------------------------------------------
# 6. Modality B: AG News with DistilBERT
# ---------------------------------------------------------
class TextDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids": torch.tensor(self.encodings["input_ids"][idx], dtype=torch.long),
            "attention_mask": torch.tensor(self.encodings["attention_mask"][idx], dtype=torch.long),
        }, int(self.labels[idx])

def evaluate_nlp(model, loader):
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for batch, y in loader:
            input_ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            out = model(input_ids=input_ids, attention_mask=mask).logits
            preds.extend(out.argmax(dim=1).cpu().numpy())
            targets.extend(y.numpy())
    return accuracy_score(targets, preds)

def run_agnews_experiment(noise_rate=0.2, noise_type="symmetric", epochs=4):
    print(f"\n--- Running AG News Experiment ({noise_rate*100:.0f}% {noise_type} noise) ---")
    dataset = load_dataset("fancyzhx/ag_news")
    tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

    train_raw = dataset["train"].shuffle(seed=42).select(range(6000))
    test_raw = dataset["test"].shuffle(seed=42).select(range(1500))
    clean_targets = train_raw["label"]
    train_enc = tokenizer(list(train_raw["text"]), truncation=True, padding=True, max_length=96)
    test_enc = tokenizer(list(test_raw["text"]), truncation=True, padding=True, max_length=96)
    noisy_labels, is_corrupted = inject_label_noise(clean_targets, 4, noise_rate, noise_type)


Running on device: cuda


In [4]:
# Define experiment parameters for standalone execution
noise_rate = 0.2
noise_type = "symmetric"
epochs = 4

print(f"--- Running AG News Experiment ({noise_rate*100:.0f}% {noise_type} noise) ---")
dataset = load_dataset("fancyzhx/ag_news")
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

train_raw = dataset["train"].shuffle(seed=42).select(range(6000))
test_raw = dataset["test"].shuffle(seed=42).select(range(1500))

train_enc = tokenizer(list(train_raw["text"]), truncation=True, padding=True, max_length=96)
test_enc = tokenizer(list(test_raw["text"]), truncation=True, padding=True, max_length=96)

clean_targets = train_raw["label"]
noisy_labels, is_corrupted = inject_label_noise(clean_targets, 4, noise_rate, noise_type)

base_train = TextDataset(train_enc, noisy_labels)
indexed_train = IndexedDataset(base_train, noisy_labels)
test_set = TextDataset(test_enc, test_raw["label"])

train_loader = DataLoader(indexed_train, batch_size=64, shuffle=True)
test_loader = DataLoader(test_set, batch_size=128, shuffle=False)

model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=4).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-5)

prob_history = np.zeros((epochs, len(indexed_train)), dtype=np.float32)

for epoch in range(epochs):
    model.train()
    for batch, y, idx in train_loader:
        input_ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        y = y.to(device)

        optimizer.zero_grad()
        logits = model(input_ids=input_ids, attention_mask=mask).logits
        loss = F.cross_entropy(logits, y)
        loss.backward()
        optimizer.step()

        probs = F.softmax(logits.detach(), dim=1)
        assigned_probs = probs[torch.arange(len(y)), y].cpu().numpy()
        prob_history[epoch, idx.numpy()] = assigned_probs

noisy_acc = evaluate_nlp(model, test_loader)
conf, var, suspicion = compute_cartography_metrics(prob_history)

auroc = roc_auc_score(is_corrupted, suspicion)
prauc = average_precision_score(is_corrupted, suspicion)
plot_cartography(conf, var, is_corrupted, f"AG News Training Dynamics ({noise_type})", "results/agnews_cartography.png")

# Prune top suspicious samples and retrain
keep_mask = np.argsort(suspicion)[:int((1.0 - noise_rate) * len(indexed_train))]
pruned_subset = Subset(indexed_train, keep_mask)
pruned_loader = DataLoader(pruned_subset, batch_size=64, shuffle=True)

clean_model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=4).to(device)
optimizer = torch.optim.AdamW(clean_model.parameters(), lr=3e-5)

for epoch in range(epochs):
    clean_model.train()
    for batch, y, _ in pruned_loader:
        input_ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        y = y.to(device)
        optimizer.zero_grad()
        loss = F.cross_entropy(clean_model(input_ids=input_ids, attention_mask=mask).logits, y)
        loss.backward()
        optimizer.step()

pruned_acc = evaluate_nlp(clean_model, test_loader)

res_nlp = {
    "Modality": "NLP (AG News)",
    "AUROC": round(auroc, 4),
    "PR-AUC": round(prauc, 4),
    "Noisy_Acc": round(noisy_acc, 4),
    "Pruned_Acc": round(pruned_acc, 4)
}
print("\n=== AG News Experiment Results ===")
print(res_nlp)


--- Running AG News Experiment (20% symmetric noise) ---


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



=== AG News Experiment Results ===
{'Modality': 'NLP (AG News)', 'AUROC': np.float64(0.9776), 'PR-AUC': np.float64(0.9261), 'Noisy_Acc': 0.8833, 'Pruned_Acc': 0.894}


In [14]:
def run_agnews_experiment(noise_rate=0.2, noise_type="symmetric", epochs=3):
    print(f"\n--- Running AG News Experiment ({noise_rate*100:.0f}% {noise_type} noise) ---")
    dataset = load_dataset("fancyzhx/ag_news")
    tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

    train_raw = dataset["train"].shuffle(seed=42).select(range(3000))
    test_raw = dataset["test"].shuffle(seed=42).select(range(1000))

    train_enc = tokenizer(list(train_raw["text"]), truncation=True, padding=True, max_length=96)
    test_enc = tokenizer(list(test_raw["text"]), truncation=True, padding=True, max_length=96)

    clean_targets = train_raw["label"]
    noisy_labels, is_corrupted = inject_label_noise(clean_targets, 4, noise_rate, noise_type)

    base_train = TextDataset(train_enc, noisy_labels)
    indexed_train = IndexedDataset(base_train, noisy_labels)
    test_set = TextDataset(test_enc, test_raw["label"])

    train_loader = DataLoader(indexed_train, batch_size=64, shuffle=True)
    test_loader = DataLoader(test_set, batch_size=128, shuffle=False)

    model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=4).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-5)

    prob_history = np.zeros((epochs, len(indexed_train)), dtype=np.float32)

    for epoch in range(epochs):
        model.train()
        for batch, y, idx in train_loader:
            input_ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            y = y.to(device)

            optimizer.zero_grad()
            logits = model(input_ids=input_ids, attention_mask=mask).logits
            loss = F.cross_entropy(logits, y)
            loss.backward()
            optimizer.step()

            probs = F.softmax(logits.detach(), dim=1)
            assigned_probs = probs[torch.arange(len(y)), y].cpu().numpy()
            prob_history[epoch, idx.numpy()] = assigned_probs

    noisy_acc = evaluate_nlp(model, test_loader)
    conf, var, suspicion = compute_cartography_metrics(prob_history)

    auroc = roc_auc_score(is_corrupted, suspicion)
    prauc = average_precision_score(is_corrupted, suspicion)
    plot_cartography(conf, var, is_corrupted, f"AG News Training Dynamics ({noise_type})", "results/agnews_cartography.png")

    keep_mask = np.argsort(suspicion)[:int((1.0 - noise_rate) * len(indexed_train))]
    pruned_subset = Subset(indexed_train, keep_mask)
    pruned_loader = DataLoader(pruned_subset, batch_size=64, shuffle=True)

    clean_model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=4).to(device)
    optimizer = torch.optim.AdamW(clean_model.parameters(), lr=3e-5)

    for epoch in range(epochs):
        clean_model.train()
        for batch, y, _ in pruned_loader:
            input_ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            y = y.to(device)
            optimizer.zero_grad()
            loss = F.cross_entropy(clean_model(input_ids=input_ids, attention_mask=mask).logits, y)
            loss.backward()
            optimizer.step()

    pruned_acc = evaluate_nlp(clean_model, test_loader)
    return {
        "Modality": "NLP (AG News)",
        "AUROC": round(float(auroc), 4),
        "PR-AUC": round(float(prauc), 4),
        "Noisy_Acc": round(float(noisy_acc), 4),
        "Pruned_Acc": round(float(pruned_acc), 4)
    }

# Execute NLP and combine with your existing res_vision
res_nlp = run_agnews_experiment(noise_rate=0.2, noise_type="symmetric", epochs=3)

# Clean float formatting for vision dict
res_vision_clean = {k: (round(float(v), 4) if isinstance(v, (float, np.floating)) else v) for k, v in res_vision.items()}

summary_df = pd.DataFrame([res_vision_clean, res_nlp])
summary_df.to_csv("results/benchmark_summary.csv", index=False)
print("\n=== Final Experiment Summary ===")
print(summary_df.to_string(index=False))



--- Running AG News Experiment (20% symmetric noise) ---


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



=== Final Experiment Summary ===
         Modality  AUROC  PR-AUC  Noisy_Acc  Pruned_Acc
Vision (CIFAR-10) 0.9365  0.8318     0.6612       0.685
    NLP (AG News) 0.9754  0.9096     0.8900       0.875


In [13]:
# Redefine run_agnews_experiment with the return statement and run it
def run_agnews_experiment(noise_rate=0.2, noise_type="symmetric", epochs=3):
    print(f"--- Running AG News Experiment ({noise_rate*100:.0f}% {noise_type} noise) ---")
    dataset = load_dataset("fancyzhx/ag_news")
    tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

    train_raw = dataset["train"].shuffle(seed=42).select(range(3000))
    test_raw = dataset["test"].shuffle(seed=42).select(range(1000))

    train_enc = tokenizer(list(train_raw["text"]), truncation=True, padding=True, max_length=96)
    test_enc = tokenizer(list(test_raw["text"]), truncation=True, padding=True, max_length=96)

    clean_targets = train_raw["label"]
    noisy_labels, is_corrupted = inject_label_noise(clean_targets, 4, noise_rate, noise_type)

    base_train = TextDataset(train_enc, noisy_labels)
    indexed_train = IndexedDataset(base_train, noisy_labels)
    test_set = TextDataset(test_enc, test_raw["label"])

    train_loader = DataLoader(indexed_train, batch_size=64, shuffle=True)
    test_loader = DataLoader(test_set, batch_size=128, shuffle=False)

    model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=4).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-5)

    prob_history = np.zeros((epochs, len(indexed_train)), dtype=np.float32)

    for epoch in range(epochs):
        model.train()
        for batch, y, idx in train_loader:
            input_ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            y = y.to(device)

            optimizer.zero_grad()
            logits = model(input_ids=input_ids, attention_mask=mask).logits
            loss = F.cross_entropy(logits, y)
            loss.backward()
            optimizer.step()

            probs = F.softmax(logits.detach(), dim=1)
            assigned_probs = probs[torch.arange(len(y)), y].cpu().numpy()
            prob_history[epoch, idx.numpy()] = assigned_probs

    noisy_acc = evaluate_nlp(model, test_loader)
    conf, var, suspicion = compute_cartography_metrics(prob_history)

    auroc = roc_auc_score(is_corrupted, suspicion)
    prauc = average_precision_score(is_corrupted, suspicion)
    plot_cartography(conf, var, is_corrupted, f"AG News Training Dynamics ({noise_type})", "results/agnews_cartography.png")

    keep_mask = np.argsort(suspicion)[:int((1.0 - noise_rate) * len(indexed_train))]
    pruned_subset = Subset(indexed_train, keep_mask)
    pruned_loader = DataLoader(pruned_subset, batch_size=64, shuffle=True)

    clean_model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=4).to(device)
    optimizer = torch.optim.AdamW(clean_model.parameters(), lr=3e-5)

    for epoch in range(epochs):
        clean_model.train()
        for batch, y, _ in pruned_loader:
            input_ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            y = y.to(device)
            optimizer.zero_grad()
            loss = F.cross_entropy(clean_model(input_ids=input_ids, attention_mask=mask).logits, y)
            loss.backward()
            optimizer.step()

    pruned_acc = evaluate_nlp(clean_model, test_loader)
    return {
        "Modality": "NLP (AG News)",
        "AUROC": round(float(auroc), 4),
        "PR-AUC": round(float(prauc), 4),
        "Noisy_Acc": round(float(noisy_acc), 4),
        "Pruned_Acc": round(float(pruned_acc), 4)
    }

# Hardcode your saved CIFAR-10 results so it never fails even if res_vision was overwritten
res_vision_saved = {
    "Modality": "Vision (CIFAR-10)",
    "AUROC": 0.9337,
    "PR-AUC": 0.8329,
    "Noisy_Acc": 0.6735,
    "Pruned_Acc": 0.6953
}

res_nlp = run_agnews_experiment(noise_rate=0.2, noise_type="symmetric", epochs=3)

summary_df = pd.DataFrame([res_vision_saved, res_nlp])
summary_df.to_csv("results/benchmark_summary.csv", index=False)
print("\n=== Final Experiment Summary ===")
print(summary_df.to_string(index=False))


--- Running AG News Experiment (20% symmetric noise) ---


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



=== Final Experiment Summary ===
         Modality  AUROC  PR-AUC  Noisy_Acc  Pruned_Acc
Vision (CIFAR-10) 0.9337  0.8329     0.6735      0.6953
    NLP (AG News) 0.9697  0.8934     0.8810      0.8750
